In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_text_splitters import RecursiveCharacterTextSplitter

from qdrant_client import QdrantClient
from qdrant_client.http import models
from qdrant_client.models import PointStruct, Distance, VectorParams

load_dotenv()
HUGGINGFACEHUB_API_TOKEN=os.getenv("HUGGINGFACEHUB_API_TOKEN")
repo_id = "Qwen/Qwen2.5-72B-Instruct"

In [2]:
def chunking(path:str, n_chunk:int = 1000, chunk_overlap:int = 100):

    loader = PyPDFLoader(path)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = n_chunk,
        chunk_overlap = chunk_overlap
    )

    documents = loader.load()
    chunks = splitter.split_documents(documents)
    chunks = [chunk.page_content for chunk in chunks]
    return chunks


path = "mongodb.pdf"
chunks = chunking(path=path)

In [3]:
import sys
sys.getsizeof(chunks)

920

In [4]:
def get_embeddings(chunks:list):

    hf_embed = HuggingFaceEmbeddings()
    embedded_chunks = hf_embed.embed_documents(chunks)
    return embedded_chunks, hf_embed

In [5]:
embedded_chunks, hf_embed = get_embeddings(chunks=chunks)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [7]:
COLLECTION_NAME = "my_collection"
BATCH_SIZE = 100

qd_client = QdrantClient(
        host = "localhost",
        port = 6333
    )

qd_client.delete_collection(collection_name=COLLECTION_NAME)

client = QdrantClient(
        host = "localhost",
        port = 6333
    )
client.create_collection(
    collection_name = "my_collection",
    vectors_config=VectorParams(size=768, distance=Distance.DOT),
)

True

In [8]:
def store_embeddings2qdrant(embedded_chunks:list):

    for batch_start in range(0, len(embedded_chunks), BATCH_SIZE):
        batch_end = batch_start + BATCH_SIZE
        batch_points = [
            PointStruct(
                id = i,
                vector= embedded_chunks[i],
                payload= {"text" : chunks[i]}
            )
            for i in range(batch_start, min(batch_end, len(embedded_chunks)))
        ]

        qd_client.upsert(
            collection_name=COLLECTION_NAME,
            wait = True,
            points= batch_points
        )
        print(f"Upserted batch {batch_start}–{min(batch_end, len(embedded_chunks))-1}")

    print(f"embedding upserted successfully")

In [9]:
store_embeddings2qdrant(embedded_chunks=embedded_chunks)

Upserted batch 0–96
embedding upserted successfully


In [12]:
def get_context(query:str):

    query_embed = hf_embed.embed_query(query)
    data = client.query_points(
        collection_name="my_collection",
        query=query_embed,
        with_payload=True,
        limit=3
    )
    context = "\n\n".join([payload.payload.get("text") for payload in data.points])
    # print(f"context: {context}")
    return context

In [13]:
question = "what problem mongodb solves? which is not solved by mysql?"
context = get_context(question)

In [15]:
def ask_llm(query:str, context:str = None):

    endpoint = HuggingFaceEndpoint(
            huggingfacehub_api_token = HUGGINGFACEHUB_API_TOKEN,
            repo_id = repo_id,
            temperature = 0.5,
            max_new_tokens = 1024
        )
    
    llm = ChatHuggingFace(llm=endpoint)
    response = llm.invoke(query)
    return response.content

MongoDB and MySQL are both powerful database systems, but they solve different types of problems due to their distinct architectures and data models. Here are some key problems that MongoDB addresses which are not as efficiently or easily handled by MySQL:

### 1. **Flexible Schema Design**
- **MongoDB**: MongoDB uses a document-oriented model, where each document can have a different structure. This flexibility allows for dynamic schemas, making it easier to handle evolving data structures without the need for complex migrations.
- **MySQL**: MySQL uses a relational model with fixed schemas. Adding or changing columns in a table can be cumbersome and may require downtime or significant overhead.

### 2. **Handling Unstructured or Semi-Structured Data**
- **MongoDB**: MongoDB excels at storing and querying unstructured or semi-structured data, such as JSON documents. This makes it ideal for applications dealing with diverse and complex data types.
- **MySQL**: MySQL is optimized for st

In [ ]:
print(ask_llm(query=question, context=context))